In [1]:
%%capture

import warnings
warnings.filterwarnings('ignore')

import altair as alt
import gcsfs
import pandas as pd

from calitp_portfolio import magics
from snapshot_utils import prep_data_utils
from snapshot_utils.project_vars import GCS_FILE_PATH

alt.data_transformers.enable("vegafusion")

ModuleNotFoundError: No module named 'prep_data_utils'

In [ ]:
#TODO add parameters cell and filter
rtpa = "Sacramento Area Council of Governments"

In [ ]:
%%capture_parameters
rtpa

# {rtpa}
## New Transit Performance Metrics

The UCLA Institute of Transportation Studies (UCLA ITS) suggests that:
>Updating the policy and legislation that governs state transit funding could help make expenditures more effective and better aligned with the state’s goals of VMT and GHG reduction, which transit can achieve only through increased ridership.

The UCLA ITS recommends using cost-efficiency metrics (operating expense per VRM/VRH/UPT) and service-effectiveness metrics (passenters per VRM/VRH) to compare transit-oriented vs. auto-oriented markets. 

The charts below display these metrics by different categories.

## Performance Metrics Explained

| Metric type          | Metric example                  | Implicit Goal(s)                       | Advantages                                   | Limitations                                  |
|----------------------|---------------------------------|---------------------------------------|----------------------------------------------|----------------------------------------------|
| Cost-efficiency     | Operating cost per revenue hour (opex_per_vrh) | Reduce costs                         | Useful in both financial and service planning | Favors high labor productivity in dense, congested areas; does not track use |
|                      | Operating cost per revenue mile (opex_per_vrm) |                                       |                                              |                                              |
|                      | Operating cost per vehicle trip (opex_per_upt) |                                       |                                              |                                              |
| Service-effectiveness| Passengers per revenue-vehicle hour (upt_per_vrh) | Increase ridership; reduce poorly patronized service | Useful for service planning; emphasizes what matters to riders | Favors high ridership; does not track costs   |
|                      | Passengers per revenue-vehicle mile (upt_per_vrm) | Increase ridership; reduce low-ridership route miles/segments | Useful for service planning                | Favors high ridership and fast vehicle speeds; does not track costs |


In [ ]:
df = pd.read_parquet(
    f"{GCS_FILE_PATH}annual.parquet",
    # should only certain columns be read in? now this table is much larger
    filesystem=gcsfs.GCSFileSystem(),
    columns = [
        "ntd_id", "source_agency", "agency_status", "source_city", 
        "year",
        "mode", "mode_full_name", "type_of_service", "type_of_service_full_name",
        "reporter_type", "reporting_module", "source_state", "primary_uza_name",
        "unlinked_passenger_trips", "vehicle_revenue_hours", "vehicle_revenue_miles",
        "operating_expenses_total",
        "opex_per_vrh", "opex_per_vrm", "opex_per_upt", 
        "upt_per_vrh", "upt_per_vrm",
        "farebox_recovery_ratio", "fare_revenue",
    ]
).pipe(
    prep_data_utils.merge_with_crosswalk
).query(
    f'rtpa_name == "{rtpa}"'
).dropna(
    subset="unlinked_passenger_trips"
)

In [ ]:
df.rtpa_name.value_counts()

In [ ]:
cost_efficiency = ["opex_per_vrh", "opex_per_vrm", "opex_per_upt"]
service_effectiveness = ["upt_per_vrh", "upt_per_vrm"]

## Agency

In [ ]:
from great_tables import GT
import gt_extras as gte
import polars as pl

def make_wide_for_nanoplot(
    df: pd.DataFrame, 
    group_cols: list, 
    value_cols: list = cost_efficiency + service_effectiveness
) -> pl.DataFrame:
    df2 = (
        df
        .sort_values(group_cols + ["year"])
        .groupby(group_cols)
        .agg({
            
            c: lambda x: list(x) for c in ["year"] + value_cols
        })
        .reset_index()
    )
    
    df_pl = pl.from_pandas(df2)

    return df_pl

In [ ]:
agency_df = prep_data_utils.calculate_efficiency_metrics_by_group(
    df, 
    ["ntd_id", "source_agency", "year", "rtpa_name"]
)
    
agency_pl = make_wide_for_nanoplot(
    agency_df, group_cols = ["ntd_id", "source_agency"]
)

In [ ]:
from great_tables import nanoplot_options

cost_nano_options=nanoplot_options(
    data_line_stroke_color="purple",
    data_area_fill_color="white", #lightsteelblue?
    data_point_fill_color="coral",
    data_point_stroke_color="white",
)

service_nano_options=nanoplot_options(
    data_line_stroke_color="steelblue", #default
    data_area_fill_color="white", #lightsteelblue?
    data_point_fill_color="darkorange",
    data_point_stroke_color="white",
)

In [ ]:
(
    GT(agency_pl)
    .cols_hide(["year"])
    .fmt_nanoplot(
        columns="opex_per_vrh", plot_type="line", missing_vals="gap", options=cost_nano_options 
    ).fmt_nanoplot(
        columns="opex_per_vrm", plot_type="line", missing_vals="gap", options=cost_nano_options
    ).fmt_nanoplot(
        columns="opex_per_upt", plot_type="line", missing_vals="gap", options=cost_nano_options 
    ).fmt_nanoplot(
        columns="upt_per_vrh", plot_type="line", missing_vals="gap", options=service_nano_options 
    ).fmt_nanoplot(
        columns="upt_per_vrm", plot_type="line", missing_vals="gap", options=service_nano_options 
    ).tab_spanner(
        label="cost-efficiency",
        columns=cost_efficiency
    ).tab_spanner(
        label="service-effectiveness",
        columns=service_effectiveness
    ).cols_label(
        ntd_id = "NTD ID",
        source_agency = "Agency",
        opex_per_vrh = "Operating Cost per VRH",
        opex_per_vrm = "Operating Cost per VRM",
        opex_per_upt = "Operating Cost per UPT",
        upt_per_vrh = "Passenger Trips per VRH",
        upt_per_vrm = "Passenger Trips per VRM",
    ).tab_options(table_font_size="14px")
)

## Mode

In [ ]:
mode_df = prep_data_utils.calculate_efficiency_metrics_by_group(
    df, 
    ["mode", "mode_full_name", "year", "rtpa_name"]
)

mode_pl = make_wide_for_nanoplot(
    mode_df, group_cols = ["mode_full_name"]
)

In [ ]:
(
    GT(mode_pl)
    .cols_hide(["year"])
    .fmt_nanoplot(
        columns="opex_per_vrh", plot_type="line", missing_vals="gap", options=cost_nano_options 
    ).fmt_nanoplot(
        columns="opex_per_vrm", plot_type="line", missing_vals="gap", options=cost_nano_options
    ).fmt_nanoplot(
        columns="opex_per_upt", plot_type="line", missing_vals="gap", options=cost_nano_options 
    ).fmt_nanoplot(
        columns="upt_per_vrh", plot_type="line", missing_vals="gap", options=service_nano_options 
    ).fmt_nanoplot(
        columns="upt_per_vrm", plot_type="line", missing_vals="gap", options=service_nano_options 
    ).tab_spanner(
        label="cost-efficiency",
        columns=cost_efficiency
    ).tab_spanner(
        label="service-effectiveness",
        columns=service_effectiveness
    ).cols_label(
        mode_full_name = "Mode",
        opex_per_vrh = "Operating Cost per VRH",
        opex_per_vrm = "Operating Cost per VRM",
        opex_per_upt = "Operating Cost per UPT",
        upt_per_vrh = "Passenger Trips per VRH",
        upt_per_vrm = "Passenger Trips per VRM",
    ).tab_options(table_font_size="14px")
)

## Type of Service

In [ ]:
tos_df = prep_data_utils.calculate_efficiency_metrics_by_group(
    df, 
    ["type_of_service", "type_of_service_full_name", "year", "rtpa_name",]
)

tos_pl = make_wide_for_nanoplot(
    tos_df, group_cols = ["type_of_service_full_name"]
)

In [ ]:
(
    GT(tos_pl)
    .cols_hide(["year"])
    .fmt_nanoplot(
        columns="opex_per_vrh", plot_type="line", missing_vals="gap", options=cost_nano_options 
    ).fmt_nanoplot(
        columns="opex_per_vrm", plot_type="line", missing_vals="gap", options=cost_nano_options
    ).fmt_nanoplot(
        columns="opex_per_upt", plot_type="line", missing_vals="gap", options=cost_nano_options 
    ).fmt_nanoplot(
        columns="upt_per_vrh", plot_type="line", missing_vals="gap", options=service_nano_options 
    ).fmt_nanoplot(
        columns="upt_per_vrm", plot_type="line", missing_vals="gap", options=service_nano_options 
    ).tab_spanner(
        label="cost-efficiency",
        columns=cost_efficiency
    ).tab_spanner(
        label="service-effectiveness",
        columns=service_effectiveness
    ).cols_label(
        type_of_service_full_name = "Type of Service",
        opex_per_vrh = "Operating Cost per VRH",
        opex_per_vrm = "Operating Cost per VRM",
        opex_per_upt = "Operating Cost per UPT",
        upt_per_vrh = "Passenger Trips per VRH",
        upt_per_vrm = "Passenger Trips per VRM",
    ).tab_options(table_font_size="14px")
)

## Cost-efficiency metrics
Cost-efficiency measures inputs to outputs: For example, the cost of operating an hour of transit service.

Per the UCLA ITS Paper
>Transit-oriented markets (which are predominantly urban), transit service tends to be relatively service-effective. But high operating costs on these (mostly) older, larger systems can inhibit efforts to improve ridership by adding service. In such contexts, assessing systems with an emphasis on **cost-efficiency (i.e., the cost of operating an hour of service)** grounds would provide incentives for agencies to **manage their costs** so as to be able to provide more service with available funding.

### Operating cost per VRH
Lower is better

This section is scatterplot of the raw values, by mode, by type_of_service
* x = raw vehicle revenue hours (log scale)
* y = operating expense total (regular linear scale)
* color = reporter_type
* args in function are a bit confusing with if/else statement (handle once and use for rest of report)
* these are side-by-side charts
* only 1 year is shown, though maybe all the years can be put onto scatterplot?

### Operating cost per VRM
Lower is better

This section is scatterplot of the raw values, by mode, by type_of_service
* x = raw vehicle revenue miles  (log scale)
* y = operating expense total (regular linear scale)
* color = reporter_type

### Operating cost per trip
Lower is better

This section is scatterplot of the raw values, by mode, by type_of_service
* x = Passenger (log scale)
* y = operating expense total (regular linear scale)
* color = reporter_type

## Service-effectiveness metrics
Service-effectiveness measures outputs to consumption: For example, passenger boardings per service hour.

Per the UCLA ITS Paper
>[In] more auto-oriented markets, transit operators tend to be relatively cost-efficient, in that they have lower operating costs but serve fewer riders. In this context, assessing systems with an emphasis on **service-effectiveness (i.e., passenger boardings per service hour)** will motivate operators to **improve ridership** by changing service hours, routes, and fares to better match local demand. Agencies might also implement fare programs with schools and other institutions, and even work with municipalities on improving land use around transit in order to increase the relative attractiveness of transit service.

### Passengers per VRH
Higher is better

This section is scatterplot of the raw values, by mode, by type_of_service
* x = vehicle revenue hours (log scale)
* y = unlinked passenger trips (regular linear scale)
* color = reporter_type

### Passengers per VRM
Higher is better

This section is scatterplot of the raw values, by mode, by type_of_service
* x = vehicle revenue miles (log scale)
* y = unlinked passenger trips (regular linear scale)
* color = reporter_type